In [ ]:
# Does upgrading only qiskit-aer-gpu (0.15.1 -> 0.17.2) fix SPSA?

!pip install -q qiskit==1.4.6 qiskit-aer-gpu==0.17.2 qiskit-algorithms==0.4.0 qiskit-optimization==0.7.0 qiskit-ibm-runtime==0.29.0 2>&1 | tail -20

print("resolved environment:")
import subprocess
for pkg in ["qiskit", "qiskit-aer-gpu", "qiskit-algorithms", "qiskit-optimization",
            "qiskit-ibm-runtime", "numpy", "scipy"]:
    out = subprocess.run(["pip", "show", pkg], capture_output=True, text=True).stdout
    ver = next((l for l in out.splitlines() if l.startswith("Version:")), "NOT FOUND")
    print(f"  {pkg}: {ver}")

print("\nimport check, watching for IBM-runtime warnings:")
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    import qiskit, qiskit_aer
    from qiskit_aer.primitives import SamplerV2 as AerSampler
    from qiskit_aer.noise import NoiseModel
    from qiskit_ibm_runtime.fake_provider import FakeCairoV2
    from qiskit_algorithms import SamplingVQE
    from qiskit_algorithms.optimizers import SPSA, COBYLA, NFT
    from qiskit.circuit.library import TwoLocal
    from qiskit.quantum_info import SparsePauliOp
    from qiskit import transpile
    import numpy as np
    cairo_nm = NoiseModel.from_backend(FakeCairoV2())
    for w in caught:
        print(f"  [{w.category.__name__}] {w.message}")
    if not caught:
        print("  (no warnings raised)")
print("qiskit:", qiskit.__version__, " qiskit_aer:", qiskit_aer.__version__)
print("noise model error channels:", len(cairo_nm.to_dict()["errors"]))

print("\nSPSA+GPU, the same case that failed before:")
rng = np.random.default_rng(0)
n_qubits = 15
terms = []
for i in range(n_qubits):
    lab = ["I"] * n_qubits; lab[i] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
for i in range(n_qubits - 1):
    lab = ["I"] * n_qubits; lab[i] = "Z"; lab[i+1] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
ising_op = SparsePauliOp.from_list(terms)

sampler = AerSampler(
    seed=42,
    options={"backend_options": {
        "noise_model": cairo_nm, "method": "statevector", "device": "GPU",
        "batched_shots_gpu": True, "batched_shots_gpu_max_qubits": 16,
        "max_parallel_threads": 0, "max_parallel_experiments": 0,
    }},
)
sampler.options.default_shots = 2000
ansatz = TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz",
                   entanglement="circular", reps=3)
ansatz_d = transpile(ansatz.decompose(reps=10), backend=sampler._backend, optimization_level=1)
init = np.random.default_rng(1).uniform(-np.pi, np.pi, size=ansatz_d.num_parameters)
try:
    vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=SPSA(maxiter=15), initial_point=init)
    result = vqe.compute_minimum_eigenvalue(ising_op)
    print(f"worked, batched_shots_gpu=True on upgraded aer-gpu: "
          f"eigenvalue={float(np.real(result.eigenvalue)):.4f}")
except Exception as exc:
    print(f"still failing: {type(exc).__name__}: {exc}")

print("\ndo COBYLA and NFT still work on the upgraded aer-gpu?")
for OptClass, name in [(COBYLA, "COBYLA"), (NFT, "NFT")]:
    try:
        vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=OptClass(maxiter=15), initial_point=init)
        result = vqe.compute_minimum_eigenvalue(ising_op)
        print(f"  {name}: SUCCESS, eigenvalue={float(np.real(result.eigenvalue)):.4f}")
    except Exception as exc:
        print(f"  {name}: failed -- regression: {type(exc).__name__}: {exc}")